# 準備と、AIを使わない最初の記録

目的：教材の場所を確認し、朝の自分の説明を保存する。目安5分、初読10分は別。**このNotebookはAPIを呼びません。**
Colabはネット接続が必要。ローカルは準備済み環境を使います。長い取得処理はハッシュを検証した準備スクリプトに分けています。

In [ ]:
from pathlib import Path
import sys
PREPARE_COLAB = False  # 新しいColabで固定版を取得する時だけTrue
RELEASE_COMMIT = "e8b02ec8bc57c197b32f78307f69944253f3a1b1"
RELEASE_SHA256 = "11b03de5459d7c34f654489c72d93b39672a10e247932221067018e7fde22322"
BOOTSTRAP_SHA256 = "f1ad167aeec1cf95e2aa835eef73c05fdeae916445985ab69679ec6060d99914"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, profile='records')
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. 共通の論文・範囲・3問を確認する

In [ ]:
from seminar_lab.config import load_yaml, material_context
from seminar_lab.records import new_record, load_records
from seminar_lab.ui import activity_view, save_pair
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')
print(activity_view(course, manifest, 'P0'))
if 'offline_record' not in globals():
    context = material_context(course, manifest, 'P0', for_input=False)
    offline_record = new_record(context)

## 2. 自分の説明を書き、保存する
`None` は前の記入を保持します。自分の文を文字列で記入して `SAVE=True` にします。

In [ ]:
annotations = {'student_explanation': None, 'evidence_location': None, 'unresolved_point': None}
SAVE = False
if SAVE:
    if any(x is not None and not isinstance(x, str) for x in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    offline_record.update({k: v for k, v in annotations.items() if v is not None})
    offline_record.update(status='manual', service='none', cost_kind='not_applicable')
    paths = save_pair([offline_record], ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == [offline_record]

## 次へ

ColabのFiles欄で`outputs`のJSONLとMarkdownをダウンロードします。再起動するとランタイム内のファイルが失われます。
朝は教員が案内するGUIを利用。APIの実行は[対話Notebook](01_dialogue_lab.ipynb)へ。